<a href="https://colab.research.google.com/github/HiveCase/DLP-DA5013/blob/main/Week1/DLP_Week1_GA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [252]:
# !pip install -U "datasets==3.6.0"

Dataset at the following URL:

https://huggingface.co/datasets/ai4bharat/naamapadam

In [253]:
from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names, concatenate_datasets, interleave_datasets
from pprint import pprint

## Question 1

In [254]:
# number of configs in dataset
configs = get_dataset_config_names('ai4bharat/naamapadam')
pprint(configs)
print(f"Number of configs in the dataset is: {len(configs)}")

['as', 'bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'or', 'pa', 'ta', 'te']
Number of configs in the dataset is: 11


In [255]:
# Number of hindi training samples
hindi_train = load_dataset('ai4bharat/naamapadam','hi', split='train')
pprint(hindi_train)
print(f"Number of Hindi training samples are: {hindi_train.num_rows}")

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 985787
})
Number of Hindi training samples are: 985787


In [256]:
tamil_data = load_dataset('ai4bharat/naamapadam','ta', split='train+test+validation')
pprint(tamil_data)
print(f"Number of dataset samples in Tamil dataset is: {tamil_data.num_rows}")

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 501435
})
Number of dataset samples in Tamil dataset is: 501435


In [257]:
print(f"Number of label classes: {hindi_train.features["ner_tags"].feature.num_classes}")
# print(f"Number of label classes: {tamil_data.features["ner_tags"].feature.num_classes}")

Number of label classes: 7


## Question 2

In [258]:
ds = load_dataset("ai4bharat/naamapadam", "ta")
pprint(ds)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 497882
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 758
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2795
    })
})


In [259]:
pprint(ds.cache_files)

{'test': [{'filename': '/root/.cache/huggingface/datasets/ai4bharat___naamapadam/ta/1.0.0/9d4f21ac57d11ed4f9ea64854fdc9f5618e61acc/naamapadam-test.arrow'}],
 'train': [{'filename': '/root/.cache/huggingface/datasets/ai4bharat___naamapadam/ta/1.0.0/9d4f21ac57d11ed4f9ea64854fdc9f5618e61acc/naamapadam-train.arrow'}],
 'validation': [{'filename': '/root/.cache/huggingface/datasets/ai4bharat___naamapadam/ta/1.0.0/9d4f21ac57d11ed4f9ea64854fdc9f5618e61acc/naamapadam-validation.arrow'}]}


## Question 3

In [260]:
import os

total_size = 0

for split in ds:
    for file in ds[split].cache_files:
        total_size += os.path.getsize(file["filename"])

print(f"Dataset size: {total_size / (1024**2):.2f} MB")

Dataset size: 179.29 MB


## Question 4

In [261]:
print(f"The number of training samples in the dataset is: {ds["train"].num_rows}")

The number of training samples in the dataset is: 497882


## Question 5

In [262]:
def num_tokens(sentence):
  return {"num_tokens" : len(sentence["tokens"])}

In [263]:
ds = ds.map(num_tokens)
pprint(ds)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'num_tokens'],
        num_rows: 497882
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'num_tokens'],
        num_rows: 758
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'num_tokens'],
        num_rows: 2795
    })
})


In [264]:
total_tokens = sum(
    sum(ds[split]["num_tokens"])
    for split in ds
)

print(f"Total tokens: {total_tokens / 1_000_000:.2f} million")

Total tokens: 6.00 million


## Question 6

In [265]:
import os

size = 0

for split in ds:
    for file in ds[split].cache_files:
        size += os.path.getsize(file["filename"])

print(f"Size after map: {size / (1024**2):.2f} MB")

Size after map: 183.14 MB


## Question 7

In [266]:
concate_ds = concatenate_datasets([ds['train'], ds['test'], ds['validation']], axis=0)
pprint(concate_ds)

Dataset({
    features: ['tokens', 'ner_tags', 'num_tokens'],
    num_rows: 501435
})


In [267]:
def sentence(example):
  return {'text': ' '.join(example["tokens"])}

In [268]:
concate_ds = concate_ds.map(sentence)
pprint(concate_ds)

Dataset({
    features: ['tokens', 'ner_tags', 'num_tokens', 'text'],
    num_rows: 501435
})


In [269]:
concate_ds = concate_ds.remove_columns(['tokens', 'ner_tags'])

In [270]:
ds = concate_ds
pprint(concate_ds)

Dataset({
    features: ['num_tokens', 'text'],
    num_rows: 501435
})


In [271]:
print(f"The number of samples in `ds` Dataset is : {concate_ds.num_rows}")

The number of samples in `ds` Dataset is : 501435


## Question 8

In [272]:
import os

print("Cache files:")

total_size = 0

for file in ds.cache_files:
    filepath = file["filename"]
    size = os.path.getsize(filepath)
    total_size += size

    print(filepath)
    print(f"Size: {size / (1024**2):.2f} MB")

print(f"\nTotal cache size: {total_size / (1024**2):.2f} MB")

Cache files:
/root/.cache/huggingface/datasets/ai4bharat___naamapadam/ta/1.0.0/9d4f21ac57d11ed4f9ea64854fdc9f5618e61acc/cache-c0c0ea782d9cefa2.arrow
Size: 296.95 MB

Total cache size: 296.95 MB


## Question 9

In [273]:
pprint(ds)

Dataset({
    features: ['num_tokens', 'text'],
    num_rows: 501435
})


In [274]:
ds1 = ds.filter(lambda x: len(x["text"].split()) >= 6)

print(ds1.num_rows)

370494


In [275]:
pprint(ds)

Dataset({
    features: ['num_tokens', 'text'],
    num_rows: 501435
})


In [276]:
ds2 = ds.filter(lambda example: len(example["text"].split())>=6)
pprint(ds2)

Dataset({
    features: ['num_tokens', 'text'],
    num_rows: 370494
})


In [277]:
pprint(ds)

Dataset({
    features: ['num_tokens', 'text'],
    num_rows: 501435
})


In [278]:
print(f"The number of sample with atleast 6 or more token are: {ds1.num_rows}")
print(f"The number of sample with atleast 6 or more token are: {ds2.num_rows}")

The number of sample with atleast 6 or more token are: 370494
The number of sample with atleast 6 or more token are: 370494


In [279]:
ds2

Dataset({
    features: ['num_tokens', 'text'],
    num_rows: 370494
})

## Question 10

Download all the splits of Tamil sub-dataset “inltkh.ta” of


https://huggingface.co/datasets/ai4bharat/indic_glue

In [280]:
ds10 = load_dataset("ai4bharat/indic_glue", "inltkh.ta")
pprint(ds10)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5346
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 669
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 669
    })
})


In [281]:
ds10 = concatenate_datasets([
    ds10["train"],
    ds10["validation"],
    ds10["test"]
])
pprint(ds10)

Dataset({
    features: ['text', 'label'],
    num_rows: 6684
})


In [282]:
ds10_filtered = ds10.filter(lambda example: len(example["text"].split())>=6)
pprint(ds10_filtered)

Dataset({
    features: ['text', 'label'],
    num_rows: 6416
})


In [283]:
ds2_text = ds2.select_columns(["text"])
ds10_text = ds10_filtered.select_columns(["text"])

In [284]:
inter_ds = interleave_datasets([ds2_text, ds10_text], probabilities=[0.8, 0.2], seed=42)
pprint(inter_ds)

Dataset({
    features: ['text'],
    num_rows: 32290
})


In [285]:
print(f"Number of Rows in the interleaved dataset is: {inter_ds.num_rows}")

Number of Rows in the interleaved dataset is: 32290
